In [ ]:
# ============================================================
# 03-SDI_Workflow.ipynb
# Suppression Difficulty Index — baseline vs. treated landscape
# ============================================================

from pathlib import Path
import pandas as pd
import rioxarray as rxr
import matplotlib.pyplot as plt

from fb_tools import (
    run_batch,
    build_scenarios,
    stacked_output_path,
    calculate_sdi,
    calculate_delta_sdi,
    fetch_osm_roads,
)

# ── Paths ─────────────────────────────────────────────────────────────────────
projdir   = Path("/Users/mcc/Library/CloudStorage/Box-Box/MCC/fire_modeling/FM_PythonWrapper")
lcp_dir   = projdir / "data/spatial/mod/fuelscapes"
fm_exe    = projdir / "code/FB/TestFlamMap.exe"
rtc_path  = projdir / "code/dev/SDI/08_RTC_lookup_SDIwt_westernUS_2021_update.txt"
out_root  = projdir / "data/spatial/mod/flammap/outputs"
sdi_dir   = projdir / "data/spatial/mod/sdi"

baseline_lcp = lcp_dir / "baseline/lcp_baseline.tif"
treated_lcp  = lcp_dir / "treated/lcp_ModSevWF.tif"

# ── 1. Define weather conditions ──────────────────────────────────────────────
conditions = pd.DataFrame({
    "Scenario":      ["Pct25", "Pct50", "Pct97"],
    "WIND_SPEED":    [8,       9,       17],
    "WIND_DIRECTION":[-1,     -1,      -1],
    "FM_1hr":        [21,      12,      5.8],
    "FM_10hr":       [17,      13,      7.5],
    "FM_100hr":      [15,      13,      8.5],
    "FM_herb":       [100,     80,      30],
    "FM_woody":      [130,     110,     60],
})

lcps = [baseline_lcp, treated_lcp]

scenarios_df = build_scenarios(
    conditions, lcps,
    outputs="FLAMELENGTH, CROWNSTATE, HEATAREA, SPREADRATE",
)
print(f"{len(scenarios_df)} total runs")
# 6 runs: 3 weather × 2 landscapes

# ── 2. Run FlamMap batch ──────────────────────────────────────────────────────
# (Skip this block if outputs already exist — stacked TIFFs are on disk)
summary = run_batch(
    fm_exe,
    scenarios_df,
    output_root=out_root,
    n_process=6,
    stack_out=True,
    cleanup=True,        # remove individual band TIFFs after stacking
)
print(summary[["Scenario", "LCP", "status"]])

# Output layout after run_batch:
#   out_root/lcp_baseline/Pct25/Pct25_LCP_BASELINE.tif   ← 4-band stack
#   out_root/lcp_baseline/Pct97/Pct97_LCP_BASELINE.tif
#   out_root/lcp_ModSevWF/Pct25/Pct25_LCP_MODSEVWF.tif
#   out_root/lcp_ModSevWF/Pct97/Pct97_LCP_MODSEVWF.tif
#   ...

# ── 3. Fetch OSM roads/trails (once per AOI) ──────────────────────────────────
ref = rxr.open_rasterio(baseline_lcp, masked=True)
aoi = ref.rio.bounds()  # returns (left, bottom, right, top) in raster CRS

import geopandas as gpd
from shapely.geometry import box
aoi_gdf = gpd.GeoDataFrame(
    geometry=[box(*aoi)],
    crs=ref.rio.crs,
)

roads_gdf, trails_gdf = fetch_osm_roads(aoi_gdf, out_crs=ref.rio.crs)
print(f"Roads: {len(roads_gdf)}  Trails: {len(trails_gdf)}")

# ── 4. Calculate SDI for each landscape × scenario ────────────────────────────
# stacked_output_path() reconstructs the path from the same args used in run_batch

SCENARIO = "Pct97"   # pick the scenario driving your SDI analysis

baseline_stack = stacked_output_path(out_root, baseline_lcp, SCENARIO)
treated_stack  = stacked_output_path(out_root, treated_lcp,  SCENARIO)

print(f"Baseline stack : {baseline_stack.name}  exists={baseline_stack.exists()}")
print(f"Treated stack  : {treated_stack.name}   exists={treated_stack.exists()}")

# --- Baseline SDI
sdi_baseline = calculate_sdi(
    lcp=baseline_lcp,
    roads_gdf=roads_gdf,
    trails_gdf=trails_gdf,
    rtc_path=rtc_path,
    flammap_stack=baseline_stack,           # ← multiband stack, no manual band extraction
    out_path=sdi_dir / f"sdi_baseline_{SCENARIO}.tif",
)

# --- Treatment SDI
sdi_treated = calculate_sdi(
    lcp=treated_lcp,
    roads_gdf=roads_gdf,
    trails_gdf=trails_gdf,
    rtc_path=rtc_path,
    flammap_stack=treated_stack,
    out_path=sdi_dir / f"sdi_ModSevWF_{SCENARIO}.tif",
)

# ── 5. Delta SDI ──────────────────────────────────────────────────────────────
delta = calculate_delta_sdi(
    sdi_baseline,
    sdi_treated,
    out_path=sdi_dir / f"delta_sdi_ModSevWF_{SCENARIO}.tif",
)
# Positive values = treatment reduced suppression difficulty

# ── 6. Quick plot ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

(sdi_baseline / 100).plot(ax=axes[0], cmap="YlOrRd", vmin=0, robust=True)
axes[0].set_title(f"Baseline SDI — {SCENARIO}")

(sdi_treated / 100).plot(ax=axes[1], cmap="YlOrRd", vmin=0, robust=True)
axes[1].set_title(f"ModSevWF SDI — {SCENARIO}")

(delta / 100).plot(ax=axes[2], cmap="RdBu", center=0, robust=True)
axes[2].set_title(f"Delta SDI (Baseline − Treated) — {SCENARIO}")

plt.tight_layout()
plt.savefig(sdi_dir / f"sdi_comparison_{SCENARIO}.png", dpi=150)
plt.show()

# ── 7. Batch SDI across all scenarios ─────────────────────────────────────────
# If you want SDI for every weather percentile:
for scenario in ["Pct25", "Pct50", "Pct97"]:
    b_stack = stacked_output_path(out_root, baseline_lcp, scenario)
    t_stack = stacked_output_path(out_root, treated_lcp,  scenario)

    sdi_b = calculate_sdi(
        lcp=baseline_lcp, roads_gdf=roads_gdf, trails_gdf=trails_gdf,
        rtc_path=rtc_path, flammap_stack=b_stack,
        out_path=sdi_dir / f"sdi_baseline_{scenario}.tif",
    )
    sdi_t = calculate_sdi(
        lcp=treated_lcp, roads_gdf=roads_gdf, trails_gdf=trails_gdf,
        rtc_path=rtc_path, flammap_stack=t_stack,
        out_path=sdi_dir / f"sdi_ModSevWF_{scenario}.tif",
    )
    calculate_delta_sdi(
        sdi_b, sdi_t,
        out_path=sdi_dir / f"delta_sdi_ModSevWF_{scenario}.tif",
    )
    print(f"[{scenario}] done")
